# 02 — A0 Magnitude U-Net Training

Fixed project root: `D:\PAPERS\SPEECH\low_snr_speech_enhancement`

This notebook reads the manifests created by Notebook 01.

In [ ]:
from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path(r"D:\PAPERS\SPEECH\low_snr_speech_enhancement")
TRAIN_MANIFEST = PROJECT_ROOT / "manifests" / "voicebank" / "train.csv"
VAL_MANIFEST   = PROJECT_ROOT / "manifests" / "voicebank" / "val.csv"
OUTPUT_DIR     = PROJECT_ROOT / "outputs" / "a0_voicebank"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_MANIFEST.exists(), f"Missing: {TRAIN_MANIFEST}"
assert VAL_MANIFEST.exists(), f"Missing: {VAL_MANIFEST}"

SEED = 42
SR = 16000
N_FFT = 512
WIN_LENGTH = 400
HOP_LENGTH = 100
SEGMENT_SECONDS = 2.56
BATCH_SIZE = 8
EPOCHS = 100
LR = 2e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 12
GRAD_CLIP = 5.0

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def read_audio(path, target_sr=16000):
    wav, sr = sf.read(path, always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = np.asarray(wav, dtype=np.float32)
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr).astype(np.float32)
    return wav

def pad_or_crop_pair(clean, noisy, target_len, training):
    n = min(len(clean), len(noisy))
    clean, noisy = clean[:n], noisy[:n]
    if n >= target_len:
        start = random.randint(0, n-target_len) if training else 0
        return clean[start:start+target_len], noisy[start:start+target_len]
    pad = target_len - n
    return np.pad(clean, (0, pad)), np.pad(noisy, (0, pad))

class PairedSpeechDataset(Dataset):
    def __init__(self, manifest, training=False):
        self.df = pd.read_csv(manifest)
        self.training = training
        self.target_len = int(SR * SEGMENT_SECONDS)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clean = read_audio(row.clean_path, SR)
        noisy = read_audio(row.noisy_path, SR)
        clean, noisy = pad_or_crop_pair(clean, noisy, self.target_len, self.training)

        peak = max(float(np.max(np.abs(clean))), float(np.max(np.abs(noisy))), 1e-8)
        if peak > 1.0:
            clean /= peak
            noisy /= peak

        return torch.from_numpy(clean), torch.from_numpy(noisy)

train_ds = PairedSpeechDataset(TRAIN_MANIFEST, training=True)
val_ds = PairedSpeechDataset(VAL_MANIFEST, training=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                          pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                        pin_memory=torch.cuda.is_available())

print("Train utterances:", len(train_ds))
print("Validation utterances:", len(val_ds))

In [ ]:
def stft_complex(waveform):
    window = torch.hann_window(WIN_LENGTH, device=waveform.device)
    return torch.stft(waveform, n_fft=N_FFT, hop_length=HOP_LENGTH,
                      win_length=WIN_LENGTH, window=window,
                      center=True, return_complex=True)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class MagnitudeUNet(nn.Module):
    def __init__(self, base_channels=16, depth=4):
        super().__init__()
        channels = [base_channels*(2**i) for i in range(depth)]
        self.encoders = nn.ModuleList()
        in_ch = 1
        for ch in channels:
            self.encoders.append(ConvBlock(in_ch, ch))
            in_ch = ch
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(channels[-1], channels[-1]*2)
        self.up_convs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        dec_in = channels[-1]*2
        for ch in reversed(channels):
            self.up_convs.append(nn.Conv2d(dec_in, ch, 1))
            self.decoders.append(ConvBlock(ch*2, ch))
            dec_in = ch
        self.out = nn.Conv2d(channels[0], 1, 1)

    def forward(self, noisy_mag):
        x = torch.log1p(noisy_mag).unsqueeze(1)
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.up_convs, self.decoders, reversed(skips)):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = up(x)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        mask = torch.sigmoid(self.out(x)).squeeze(1)
        return mask * noisy_mag, mask

model = MagnitudeUNet().to(device)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def reconstruction_loss(enhanced_mag, clean_mag):
    return F.l1_loss(torch.log1p(enhanced_mag), torch.log1p(clean_mag))

@torch.no_grad()
def validate():
    model.eval()
    total = 0.0
    count = 0
    for clean, noisy in val_loader:
        clean = clean.to(device, non_blocking=True)
        noisy = noisy.to(device, non_blocking=True)
        clean_mag = stft_complex(clean).abs()
        noisy_mag = stft_complex(noisy).abs()
        enhanced_mag, _ = model(noisy_mag)
        loss = reconstruction_loss(enhanced_mag, clean_mag)
        total += loss.item() * clean.size(0)
        count += clean.size(0)
    return total / max(count, 1)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_val = float("inf")
bad_epochs = 0
history = []

for epoch in range(1, EPOCHS+1):
    model.train()
    running = 0.0
    seen = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for clean, noisy in pbar:
        clean = clean.to(device, non_blocking=True)
        noisy = noisy.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            clean_mag = stft_complex(clean).abs()
            noisy_mag = stft_complex(noisy).abs()
            enhanced_mag, _ = model(noisy_mag)
            loss = reconstruction_loss(enhanced_mag, clean_mag)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running += loss.item() * clean.size(0)
        seen += clean.size(0)
        pbar.set_postfix(train_loss=running/max(seen, 1))

    train_loss = running / max(seen, 1)
    val_loss = validate()

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "history.csv", index=False)

    state = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_loss": val_loss,
        "seed": SEED
    }
    torch.save(state, OUTPUT_DIR / "last.pt")

    if val_loss < best_val:
        best_val = val_loss
        bad_epochs = 0
        torch.save(state, OUTPUT_DIR / "best.pt")
    else:
        bad_epochs += 1

    print(f"Epoch {epoch}: train={train_loss:.6f}, val={val_loss:.6f}, best={best_val:.6f}")

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

print("Training complete.")